# ProPublica Nonprofit Explorer API

Pulls data from the [ProPublica Nonprofit Explorer API](https://projects.propublica.org/nonprofits/api).

**No API key required.**


In [4]:
import json
import requests
import pandas as pd
from pathlib import Path
from utility import propublica_endpoint


Set up basic folder structure

In [5]:
curr_dir = Path()
pp_dir = curr_dir / 'ProPublica'
pp_input_dir = pp_dir / 'input'
pp_output_dir = pp_dir / 'output'
pp_input_dir.mkdir(parents=True, exist_ok=True)
pp_output_dir.mkdir(parents=True, exist_ok=True)


## Overview of root endpoint categories

In [6]:
import pprint
# ProPublica API has no root catalog endpoint. Show available top-level keys from the search endpoint instead.
r = requests.get(propublica_endpoint('search'), params={'q': 'social welfare'}).json()
pprint.pprint({k: v for k, v in r.items() if k != 'organizations'})


{'api_version': 2,
 'cur_page': 0,
 'data_source': 'ProPublica Nonprofit Explorer API: '
                'https://projects.propublica.org/nonprofits/api/\n'
                'IRS Exempt Organizations Business Master File Extract (EO '
                'BMF): '
                'https://www.irs.gov/charities-non-profits/exempt-organizations-business-master-file-extract-eo-bmf\n'
                'IRS Annual Extract of Tax-Exempt Organization Financial Data: '
                'https://www.irs.gov/uac/soi-tax-stats-annual-extract-of-tax-exempt-organization-financial-data',
 'num_pages': 2,
 'page_offset': 0,
 'per_page': 25,
 'search_query': 'social welfare',
 'selected_code': None,
 'selected_ntee': None,
 'selected_state': None,
 'total_results': 39}


## Search

Search for nonprofit organizations by keyword. Returns basic org info: EIN, name, state, NTEE code, revenue.

### Generate examples

In [7]:
search_terms = ['political advocacy', 'social welfare', 'dark money']
search_results = []

for term in search_terms:
    response = requests.get(
        propublica_endpoint('search'),
        params={'q': term}
    ).json()
    orgs = response.get('organizations', [])
    for org in orgs:
        org['search_term'] = term
    search_results += orgs
    print(f"'{term}': {len(orgs)} results")

print(f'Total: {len(search_results)} organizations')


'political advocacy': 4 results
'social welfare': 25 results
'dark money': 0 results
Total: 29 organizations


In [8]:
pp_search_path = pp_input_dir / 'propublica_search_sample.json'
with open(pp_search_path, mode='w') as f:
    f.write(json.dumps(search_results, indent=4))


### Parse examples

In [9]:
with open(pp_search_path) as f:
    search_sample = json.load(f)
search_df = pd.DataFrame(search_sample)
search_df.head()


,ein,strein,name,sub_name,city,state,ntee_code,raw_ntee_code,subseccd,has_subseccd,have_filings,have_extracts,have_pdfs,score,search_term
0,334924099,33-4924099,Muslim American International Political Advoca...,Muslim American International Political Advoca...,Washington,DC,Q01,Q01,4.0,True,None,None,None,85.589700,political advocacy
1,992273299,99-2273299,Connecticut Youth Political Advocacy Center Inc,Connecticut Youth Political Advocacy Center Inc,Westbrook,CT,O99,O99,4.0,True,None,None,None,85.589700,political advocacy
2,831526970,83-1526970,Women For Political Change Education & Advocac...,Women For Political Change Education & Advocac...,Saint Paul,MN,NaN,NaN,3.0,True,None,None,None,79.338005,political advocacy
3,932991361,93-2991361,Urban Political Pipeline & Advocacy Network Fo...,Urban Political Pipeline & Advocacy Network Fo...,Washington,DC,R99,R99,4.0,True,None,None,None,79.338005,political advocacy
4,616034264,61-6034264,Kentucky Social Welfare Foundation,Kentucky Social Welfare Foundation,Union,KY,NaN,NaN,3.0,True,None,None,None,83.238250,social welfare


In [10]:
search_df.to_csv(pp_output_dir / 'propublica_search_sample.csv')


## Organization Detail

Full organization profile by EIN, including financial history summary.

### Generate examples

In [11]:
sample_eins = search_df['ein'].dropna().unique()[:10].tolist()
org_details = []

for ein in sample_eins:
    response = requests.get(propublica_endpoint('organization', str(int(ein)))).json()
    org = response.get('organization', {})
    if org:
        org_details.append(org)
    print(f"EIN {ein}: {org.get('name', 'N/A')}")


EIN 334924099: Muslim American International Political Advocacy Council
EIN 992273299: Connecticut Youth Political Advocacy Center Inc
EIN 831526970: Women For Political Change Education & Advocacy Fund
EIN 932991361: Urban Political Pipeline & Advocacy Network Foundation Inc
EIN 616034264: Kentucky Social Welfare Foundation
EIN 273558461: Questa Social Welfare Association
EIN 20777334: Siyon Social Welfare Society
EIN 510432646: Social Welfare Activities Usa
EIN 237248967: Episcopal Social Welfare Council
EIN 874672227: Honest Social Welfare Organization


In [12]:
pp_org_path = pp_input_dir / 'propublica_org_detail_sample.json'
with open(pp_org_path, mode='w') as f:
    f.write(json.dumps(org_details, indent=4))


### Parse examples

In [13]:
with open(pp_org_path) as f:
    org_sample = json.load(f)
org_df = pd.DataFrame(org_sample)
org_df.head()


,id,ein,name,careofname,address,city,state,zipcode,exemption_number,subsection_code,...,income_amount,revenue_amount,ntee_code,sort_name,created_at,updated_at,data_source,have_extracts,have_pdfs,latest_object_id
0,334924099,334924099,Muslim American International Political Advoca...,NaN,1717 N ST NW STE 1,Washington,DC,20036-2827,0,4,...,0.0,0.0,Q01,None,2026-02-19T15:32:02.169Z,2026-04-15T13:40:51.676Z,current_2026_04_15,None,None,NaN
1,992273299,992273299,Connecticut Youth Political Advocacy Center Inc,% RYAN M ENGELS,90 FISHING BROOK RD,Westbrook,CT,06498-1469,0,4,...,0.0,0.0,O99,None,2025-11-18T01:37:42.741Z,2026-04-15T13:39:56.571Z,current_2026_04_15,None,None,NaN
2,831526970,831526970,Women For Political Change Education & Advocac...,NaN,2380 WYCLIFF ST B 103,Saint Paul,MN,55114-1257,0,3,...,638955.0,638673.0,NaN,None,2023-05-09T20:33:57.968Z,2024-07-22T20:22:04.620Z,pre_2024_08_27,None,None,202133199349329493
3,932991361,932991361,Urban Political Pipeline & Advocacy Network Fo...,% THE URBAN HEALTH CONSULTANT,1930 18TH ST NW STE B2 PMB 2004,Washington,DC,20009-1797,0,4,...,0.0,0.0,R99,None,2024-08-27T13:34:16.890Z,2026-04-15T13:43:23.282Z,current_2026_04_15,None,None,NaN
4,616034264,616034264,Kentucky Social Welfare Foundation,NaN,PO BOX 1288,Union,KY,41091-1288,0,3,...,3115848.0,NaN,NaN,None,2023-05-09T20:33:05.085Z,2026-04-15T13:42:30.787Z,current_2026_04_15,None,None,202502799349100505


In [14]:
org_df.to_csv(pp_output_dir / 'propublica_org_detail_sample.csv')


## Filings

990 filings with structured data for each sampled organization. Each record includes tax year, revenue, expenses, assets, and a PDF URL.

### Generate examples

In [15]:
filings_data = []

for ein in sample_eins:
    response = requests.get(propublica_endpoint('organization', str(int(ein)))).json()
    filings = response.get('filings_with_data', [])
    for filing in filings:
        filing['ein'] = ein
    filings_data += filings
    print(f"EIN {ein}: {len(filings)} filings with data")


EIN 334924099: 0 filings with data
EIN 992273299: 0 filings with data
EIN 831526970: 2 filings with data
EIN 932991361: 0 filings with data
EIN 616034264: 10 filings with data
EIN 273558461: 0 filings with data
EIN 20777334: 6 filings with data
EIN 510432646: 0 filings with data
EIN 237248967: 0 filings with data
EIN 874672227: 0 filings with data


In [16]:
pp_filings_path = pp_input_dir / 'propublica_filings_sample.json'
with open(pp_filings_path, mode='w') as f:
    f.write(json.dumps(filings_data, indent=4))


### Parse examples

In [17]:
with open(pp_filings_path) as f:
    filings_sample = json.load(f)
filings_df = pd.DataFrame(filings_sample)
filings_df.head()


,tax_prd,tax_prd_yr,formtype,pdf_url,updated,totrevenue,totfuncexpns,totassetsend,totliabend,pct_compnsatncurrofcr,...,pubsuprtcola,pubsuprtcolb,pubsuprtcolc,pubsuprtcold,pubsuprttot,grsinvstinca,grsinvstincb,grsinvstincc,grsinvstincd,grsinvstinctot
0,202012,2020,0,NaN,2022-07-01T20:38:50.673Z,638673,701844,314002,324600,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201912,2019,1,https://projects.propublica.org/nonprofits/dow...,2022-04-05T21:25:01.990Z,85848,33286,52562,0,-0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,202312,2023,2,NaN,2025-08-05T16:14:39.460Z,336397,336978,2376872,1,-0.001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,202212,2022,2,https://projects.propublica.org/nonprofits/dow...,2024-08-28T20:28:27.624Z,202670,344811,2158602,1,-0.001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,202112,2021,2,https://projects.propublica.org/nonprofits/dow...,2024-08-28T20:02:21.645Z,375564,200643,2912037,1,-0.001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
filings_df.to_csv(pp_output_dir / 'propublica_filings_sample.csv')
